# Tugas 1 — Implementasi Algoritma Apriori (Aturan Asosiasi)

**Mata Kuliah:** Data Mining

## Deskripsi
Program untuk menganalisis aturan asosiasi dari **10 transaksi** dengan **5 variasi produk**: `susu`, `gula`, `teh`, `roti`, `kopi`.

Hal yang dihitung:
1. **F1** — frekuensi setiap item (1-itemset).
2. **F2** — semua kombinasi 2-itemset. Dari 5 produk = $\binom{5}{2} = 10$ kombinasi.
3. **F3** — semua kombinasi 3-itemset. Dari 5 produk = $\binom{5}{3} = 10$ kombinasi.
4. **Support** = frekuensi itemset / total transaksi (10).
5. **Confidence** = support(A∪B) / support(A).
6. **Nilai Final** = support × confidence (untuk ranking aturan).

## 1. Import Library dan Data Transaksi

### Penjelasan kode

- `from itertools import combinations` dipakai untuk men-generate semua kombinasi itemset (F2, F3) tanpa pengulangan.
- `pandas` dipakai untuk menampilkan tabel hasil.
- `transaksi` adalah `dict` berisi 10 transaksi pasar swalayan; setiap value adalah `set` agar pengecekan keanggotaan cepat.
- `produk` = daftar 5 item unik (susu, gula, teh, roti, kopi).
- `TOTAL_TRX = 10` dipakai sebagai penyebut saat menghitung **support**.
- `df_trx` hanyalah representasi DataFrame supaya 10 transaksi bisa ditampilkan rapi.

In [1]:
from itertools import combinations
import pandas as pd

# 10 transaksi pasar swalayan (sumber: Tabel 1 Record Transaksi)
# Produk: susu, gula, teh, roti, kopi
transaksi = {
    1:  {'susu', 'gula', 'teh'},
    2:  {'teh', 'gula', 'roti'},
    3:  {'teh', 'gula'},
    4:  {'susu', 'roti'},
    5:  {'susu', 'gula', 'roti'},
    6:  {'teh', 'gula'},
    7:  {'gula', 'kopi', 'susu'},
    8:  {'gula', 'kopi', 'susu'},
    9:  {'susu', 'roti', 'kopi'},
    10: {'gula', 'teh', 'kopi'},
}

produk = ['susu', 'gula', 'teh', 'roti', 'kopi']
TOTAL_TRX = len(transaksi)

df_trx = pd.DataFrame([
    {'TID': tid, 'Items': ', '.join(sorted(items))}
    for tid, items in transaksi.items()
])
df_trx

,TID,Items
0,1,"gula, susu, teh"
1,2,"gula, roti, teh"
2,3,"gula, teh"
3,4,"roti, susu"
4,5,"gula, roti, susu"
5,6,"gula, teh"
6,7,"gula, kopi, susu"
7,8,"gula, kopi, susu"
8,9,"kopi, roti, susu"
9,10,"gula, kopi, teh"


## 2. Fungsi Bantu: Hitung Support

### Penjelasan kode

Dua fungsi bantu untuk seluruh perhitungan Apriori:

- `hitung_frekuensi(itemset)` — menghitung berapa transaksi yang **memuat seluruh item** dalam `itemset`. Memakai `set.issubset` agar pengecekan O(1) per item.
- `support(itemset)` — frekuensi dibagi total transaksi → nilai 0..1.

Rumus: $\text{Support}(X) = \dfrac{\text{jumlah transaksi yang memuat } X}{\text{total transaksi}}$.

In [2]:
def hitung_frekuensi(itemset):
    """Berapa transaksi yang memuat seluruh item dalam itemset."""
    s = set(itemset)
    return sum(1 for items in transaksi.values() if s.issubset(items))

def support(itemset):
    return hitung_frekuensi(itemset) / TOTAL_TRX

## 3. F1 — 1-itemset

### Penjelasan kode

- Looping 5 produk, untuk tiap item dihitung frekuensi & support 1-itemset.
- Hasilnya disusun ke `df_F1` (5 baris, satu baris per item).
- Dari sini terlihat **gula** paling sering muncul (8/10), sedangkan **roti** dan **kopi** paling jarang (4/10).

In [3]:
F1 = []
for item in produk:
    f = hitung_frekuensi([item])
    F1.append({'Itemset': item, 'Frekuensi': f, 'Support': f / TOTAL_TRX})

df_F1 = pd.DataFrame(F1)
df_F1

,Itemset,Frekuensi,Support
0,susu,6,0.6
1,gula,8,0.8
2,teh,5,0.5
3,roti,4,0.4
4,kopi,4,0.4


## 4. F2 — 2-itemset (10 kombinasi)

Dari 5 produk, jumlah kombinasi 2-itemset = $\binom{5}{2} = 10$. Slide kelas hanya menampilkan 6, jadi di sini sengaja **dihitung lengkap**.

### Penjelasan kode

- `combinations(produk, 2)` menghasilkan **10 pasangan unik** (tanpa pengulangan dan tanpa urutan) — sesuai $\binom{5}{2}=10$.
- Untuk setiap pasangan, dihitung berapa transaksi memuat **kedua** item lalu support-nya.
- `print(...)` dipakai sebagai sanity check bahwa jumlah kombinasi tepat 10 (slide kelas hanya menampilkan 6).

In [4]:
F2 = []
for combo in combinations(produk, 2):
    f = hitung_frekuensi(combo)
    F2.append({
        'Itemset': ' & '.join(combo),
        'Frekuensi': f,
        'Support': f / TOTAL_TRX,
    })

df_F2 = pd.DataFrame(F2)
print(f'Jumlah kombinasi F2: {len(df_F2)}  (seharusnya 10)')
df_F2

Jumlah kombinasi F2: 10  (seharusnya 10)


,Itemset,Frekuensi,Support
0,susu & gula,4,0.4
1,susu & teh,1,0.1
2,susu & roti,3,0.3
3,susu & kopi,3,0.3
4,gula & teh,5,0.5
5,gula & roti,2,0.2
6,gula & kopi,3,0.3
7,teh & roti,1,0.1
8,teh & kopi,1,0.1
9,roti & kopi,1,0.1


## 5. F3 — 3-itemset (10 kombinasi)

Dari 5 produk, jumlah kombinasi 3-itemset = $\binom{5}{3} = 10$.

### Penjelasan kode

- Sama seperti F2 tetapi ukuran kombinasi = 3 → menghasilkan $\binom{5}{3}=10$ kombinasi.
- Beberapa kombinasi memiliki frekuensi **0** (mis. `susu & teh & roti`) artinya tidak pernah muncul bersamaan dalam 10 transaksi.

In [5]:
F3 = []
for combo in combinations(produk, 3):
    f = hitung_frekuensi(combo)
    F3.append({
        'Itemset': ' & '.join(combo),
        'Frekuensi': f,
        'Support': f / TOTAL_TRX,
    })

df_F3 = pd.DataFrame(F3)
print(f'Jumlah kombinasi F3: {len(df_F3)}  (seharusnya 10)')
df_F3

Jumlah kombinasi F3: 10  (seharusnya 10)


,Itemset,Frekuensi,Support
0,susu & gula & teh,1,0.1
1,susu & gula & roti,1,0.1
2,susu & gula & kopi,2,0.2
3,susu & teh & roti,0,0.0
4,susu & teh & kopi,0,0.0
5,susu & roti & kopi,1,0.1
6,gula & teh & roti,1,0.1
7,gula & teh & kopi,1,0.1
8,gula & roti & kopi,0,0.0
9,teh & roti & kopi,0,0.0


## 6. Aturan Asosiasi (Confidence & Nilai Final)

Untuk setiap itemset di F2 dan F3, dibentuk semua aturan **A → B** dimana A∪B = itemset.

- **Support(A→B)** = support(A∪B)
- **Confidence(A→B)** = support(A∪B) / support(A)
- **Final** = Support × Confidence

### Penjelasan kode

Fungsi `buat_aturan(itemset)` menghasilkan semua aturan **A → B** dari sebuah itemset:

- Loop `r` dari 1 sampai `len(items)-1` → ukuran sisi kiri (A) berbeda-beda.
- Untuk setiap pilihan A dengan `combinations`, sisi kanan B adalah sisa item.
- Menghitung tiga metrik:
  - **Support(A→B)** = support gabungan A∪B,
  - **Confidence(A→B)** = support(A∪B) / support(A),
  - **Final** = Support × Confidence (dipakai untuk meranking).

Fungsi diterapkan ke semua kombinasi F2 dan F3, hasilnya digabung lalu diurut **descending** berdasarkan kolom `Final`.

In [6]:
def buat_aturan(itemset):
    """Generate semua aturan A -> B dari sebuah itemset."""
    items = list(itemset)
    aturan = []
    for r in range(1, len(items)):
        for A in combinations(items, r):
            B = tuple(sorted(set(items) - set(A)))
            sup_AB = support(items)
            sup_A = support(A)
            conf = sup_AB / sup_A if sup_A > 0 else 0
            aturan.append({
                'A': ', '.join(sorted(A)),
                'B': ', '.join(B),
                'Support': sup_AB,
                'Confidence': conf,
                'Final': sup_AB * conf,
            })
    return aturan

rules = []
for combo in combinations(produk, 2):
    rules += buat_aturan(combo)
for combo in combinations(produk, 3):
    rules += buat_aturan(combo)

df_rules = pd.DataFrame(rules)
df_rules = df_rules.sort_values('Final', ascending=False).reset_index(drop=True)
df_rules

,A,B,Support,Confidence,Final
0,teh,gula,0.5,1.000000,0.500000
1,gula,teh,0.5,0.625000,0.312500
2,susu,gula,0.4,0.666667,0.266667
3,roti,susu,0.3,0.750000,0.225000
4,kopi,susu,0.3,0.750000,0.225000
...,...,...,...,...,...
75,roti,"kopi, teh",0.0,0.000000,0.000000
76,kopi,"roti, teh",0.0,0.000000,0.000000
77,"roti, teh",kopi,0.0,0.000000,0.000000
78,"kopi, teh",roti,0.0,0.000000,0.000000


## 7. Ranking Top-10 Aturan Asosiasi

### Penjelasan kode

- Mengambil 10 aturan teratas dari `df_rules` (sudah terurut).
- Menambahkan kolom `Aturan` dengan format `A → B` agar lebih mudah dibaca.
- Tabel ini adalah jawaban inti dari tugas: aturan asosiasi terkuat berdasarkan Support × Confidence.

In [7]:
top10 = df_rules.head(10).copy()
top10['Aturan'] = top10['A'] + ' → ' + top10['B']
top10[['Aturan', 'Support', 'Confidence', 'Final']]

,Aturan,Support,Confidence,Final
0,teh → gula,0.5,1.000000,0.500000
1,gula → teh,0.5,0.625000,0.312500
2,susu → gula,0.4,0.666667,0.266667
3,roti → susu,0.3,0.750000,0.225000
4,kopi → susu,0.3,0.750000,0.225000
5,kopi → gula,0.3,0.750000,0.225000
6,gula → susu,0.4,0.500000,0.200000
7,susu → roti,0.3,0.500000,0.150000
8,susu → kopi,0.3,0.500000,0.150000
9,"kopi, susu → gula",0.2,0.666667,0.133333


## 8. Kesimpulan

- F2 menghasilkan **10 kombinasi** lengkap (sesuai $\binom{5}{2}$), bukan 6 seperti di slide.
- F3 menghasilkan **10 kombinasi** lengkap (sesuai $\binom{5}{3}$).
- Nilai **Final = Support × Confidence** dipakai untuk meranking aturan: aturan dengan Final tertinggi merupakan aturan asosiasi terkuat dari data ini.
- Aturan teratas dapat dipakai sebagai dasar rekomendasi *cross-selling* (mis. jika pelanggan beli A → kemungkinan besar membeli B).